# 35 · Checkpoint: Scout, a research assistant in ~150 lines
Level L3 · Part 4 · core · ~120 min · builds on 26, 27, 28, 29, 33

## Goal
Build **Scout v1**, the first tool-using Scout, as one small plain-Python agent: a tool registry,
three tools (search the notes, calculate, write a file), a loop with a step limit, and a Gemini model
behind it, in about 150 lines and no framework. Then pass the exit test: add an extra tool, `read_file`, without touching the loop,
and show Scout recovering when a tool raises.

## The idea
Part 4 built each piece on its own: the loop (lesson 26), the registry (27), error results (28) and
good tools (29). This checkpoint joins them. Scout is an agent, not a workflow (lesson 33): questions
vary, so the model picks the steps. Stopping when the steps run out is just lesson 26's `max_steps`.

The exit test checks two properties. The loop is **open to new tools**: a new tool is one decorated
function, and the loop's code does not change. It is **robust**: a tool that raises becomes a message
the model reads, not a crash.

Analogy: a power strip. You plug in a new device without rewiring the house, and a blown fuse trips
one socket, not the whole building.

## Picture
```mermaid
flowchart LR
    Q["question"] --> L["run_agent<br/>(max_steps)"]
    L --> M["model<br/>(fake or Gemini)"]
    M -- "tool calls" --> T["run_tool<br/>(never raises, capped)"]
    T --> R[("registry<br/>search_notes, calculator,<br/>write_file, + read_file")]
    T -- "result or error" --> L
    M -- "no tool calls" --> A["answer"]
```

## Requirements
- [ ] a `ToolRegistry` whose `tool` decorator builds each schema from type hints and the docstring, and
  keeps `functions` (name → function) and `schemas` (lesson 27);
- [ ] `search_notes(query, max_results=5, page=1)` with pages and shortened texts, `calculator(expression)`
  that never calls `eval`, and `write_file(filename, text, overwrite=False)` that cannot leave Scout's
  folder (29). The source project says web search. Scout searches local notes instead, so the notebook
  runs with no search key (RESEARCH.md: a live search needs a provider key);

- [ ] `run_tool(call, tools)` that never raises and cuts big results (lessons 28, 29);
- [ ] `run_agent(question, model, tools, schemas, max_steps=8)` returning `answer`, `stop`, `steps`,
  `errors` and `messages` (lesson 26);
- [ ] a `GeminiModel` with the fake's call shape that builds its tool list on every call.

Done when `check_scout(run_agent, registry)` prints `all checks passed` and the exit test below passes.
All tool names and signatures must match exactly.

### Exact names the checks rely on
- Tools: `search_notes(query, max_results=5, page=1)`, `calculator(expression)`,
  `write_file(filename, text, overwrite=False)`; save files under the global `SCOUT_DIR`.
- `run_tool(call, tools) -> {"content", "is_error"}`. For an unknown tool, the error text lists the real
  tool names.
- `run_agent` returns `answer`, `stop`, `steps`, `errors`, `messages`; `stop` is exactly `"end_of_turn"`
  or `"step_limit"`.
- Messages: user `{"role", "text"}`; assistant `{"role", "tool_calls": [{"id", "name", "args"}]}` or
  `{"role", "text"}`; tool `{"role": "tool", "call_id", "name", "content", "is_error"}`.
- `messages[2]` is the first tool result.
- `ScriptedModel` rejects a tool message whose `call_id` it never issued.

## Build it
### Starter code: the recorder, the client and the notes
Every Gemini call is recorded once to `../data/llm_cassettes/35_checkpoint-research-assistant.yaml`
and replayed after that, so the notebook runs with no key.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("35_checkpoint-research-assistant")

import ast, contextlib, hashlib, inspect, io, json, operator, os, re
from collections.abc import Callable
from pathlib import Path
from typing import Annotated, get_args, get_origin, get_type_hints
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
print("model:", MODEL)

LLM mode: replay (35_checkpoint-research-assistant.yaml)


model: gemini-3.5-flash-lite


Scout's notes as lesson 29 left them: the six facts, a long briefing and a 30-day habitat log. Scout's
folder starts empty on every run.

In [2]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
BRIEFING = {"id": 7, "text": "Mars mission briefing: " + "check suits, check air, check water. " * 40}
LOG = [{"id": 100 + d, "text": f"Mars habitat log, day {d}: routine checks, all systems normal."}
       for d in range(1, 31)]
ALL_NOTES = NOTES + [BRIEFING] + LOG

SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)
def clear_folder():
    for old in SCOUT_DIR.glob("*.txt"):
        old.unlink()
clear_folder()
print(len(ALL_NOTES), "notes | Scout's folder:", sorted(p.name for p in SCOUT_DIR.iterdir()))

37 notes | Scout's folder: []


### Starter code: the scripted fake model (lesson 26)
Unchanged. It plays a script, gives each tool call an id, and rejects a tool result for a call it
never made.

In [3]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

print(ScriptedModel([{"text": "hello"}])([{"role": "user", "text": "hi"}], []))

{'role': 'assistant', 'text': 'hello'}


### Starter code: the acceptance scripts
Five fake runs. The happy path uses all three tools. The others hit an error and recover, call a tool
that does not exist, try to write outside the folder, and never stop.

In [4]:
QUESTION = "How many minutes long is a day on Mars? Save the answer to mars_day.txt."
def last_result(messages): return json.loads(messages[-1]["content"])
def calc(expr): return {"tool_calls": [{"name": "calculator", "args": {"expression": expr}}]}

HAPPY = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]}, calc("24*60+39"),
         lambda m: {"tool_calls": [{"name": "write_file", "args": {
             "filename": "mars_day.txt", "text": f"A day on Mars lasts about {last_result(m)} minutes."}}]},
         {"text": "A day on Mars lasts about 1479 minutes. Saved to mars_day.txt."}]
RECOVER = [calc("2^10"), calc("2**10"), lambda m: {"text": f"2 to the power 10 is {last_result(m)}."}]
UNKNOWN = [{"tool_calls": [{"name": "web_search", "args": {"query": "Mars"}}]}, {"text": "I can't search the web."}]
ESCAPE = [{"tool_calls": [{"name": "write_file", "args": {"filename": "../escaped.txt", "text": "x"}}]},
          {"text": "I can only save inside my folder."}]
STUBBORN = [{"tool_calls": [{"name": "search_notes", "args": {"query": "Pluto"}}]}]
print("scripts:", [len(s) for s in (HAPPY, RECOVER, UNKNOWN, ESCAPE, STUBBORN)], "replies each")

scripts: [4, 3, 2, 2, 1] replies each


### Starter code: the checks
`check_scout` runs each script through your loop with the trace hidden, and asserts what must be true.

In [5]:
def check_scout(run_agent, registry) -> None:
    tools, schemas = registry.functions, registry.schemas
    names = [s["name"] for s in schemas]
    assert {"search_notes", "calculator", "write_file"} <= set(names), f"registered: {names}"
    clear_folder()
    def run(script, question="?", **kw):
        with contextlib.redirect_stdout(io.StringIO()):
            return run_agent(question, ScriptedModel(script), tools, schemas, **kw)
    r = run(HAPPY, QUESTION)
    assert r["stop"] == "end_of_turn" and r["errors"] == 0, "happy path should end cleanly"
    assert "1479" in (SCOUT_DIR / "mars_day.txt").read_text(), "write_file should save the answer"
    r = run(RECOVER)
    assert r["errors"] == 1 and "1024" in r["answer"], "a raising tool should become an error result"
    r = run(UNKNOWN)
    assert r["messages"][2]["is_error"] and "search_notes" in r["messages"][2]["content"], "list real tools"
    r = run(ESCAPE)
    assert r["errors"] == 1 and not Path("../data/escaped.txt").exists(), "write_file escaped its folder"
    r = run(STUBBORN, max_steps=3)
    assert r["stop"] == "step_limit" and r["steps"] == 3 and r["answer"] is None, "step limit"
    clear_folder()
    print("all checks passed")

### Your turn
Write the registry, the three tools, `run_tool` and `run_agent`, then run
`check_scout(run_agent, registry)`. Then add a `GeminiModel` and ask Scout the Mars question for real.
Reuse lessons 26 to 29 freely: copying working parts is the point.

In [6]:
# your code here

## Reference solution
Try it yourself first. The steps below are one way to meet every requirement.

### R1. The registry (lesson 27)
Hints become JSON Schema types, parameters without a default are required, and the docstring becomes
the description. The decorator stores the schema and the function under the same name.

In [7]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean"}

def json_type(hint) -> dict:
    if hint in JSON_TYPES:
        return {"type": JSON_TYPES[hint]}
    if get_origin(hint) is list:
        return {"type": "array", "items": json_type(get_args(hint)[0])}
    raise TypeError(f"no JSON Schema type for {hint!r}")

def schema_from_function(fn) -> dict:
    hints = get_type_hints(fn, include_extras=True)
    properties, required = {}, []
    for name, param in inspect.signature(fn).parameters.items():
        if name not in hints:
            raise TypeError(f"{fn.__name__}: parameter {name!r} has no type hint")
        hint, note = hints[name], None
        if get_origin(hint) is Annotated:
            hint, note = get_args(hint)[0], get_args(hint)[1]
        properties[name] = json_type(hint) | ({"description": note} if note else {})
        if param.default is inspect.Parameter.empty:
            required.append(name)
    if not inspect.getdoc(fn):
        raise TypeError(f"{fn.__name__}: needs a docstring; it becomes the tool description")
    return {"name": fn.__name__, "description": inspect.getdoc(fn),
            "parameters": {"type": "object", "properties": properties, "required": required}}

The registry itself. It refuses a duplicate name, so two tools can never shadow each other.

In [8]:
class ToolRegistry:
    def __init__(self):
        self.functions: dict[str, Callable] = {}             # the dispatch table
        self.schemas: list[dict] = []

    def tool(self, fn):
        schema = schema_from_function(fn)
        if schema["name"] in self.functions:
            raise ValueError(f"tool {schema['name']!r} is already registered")
        self.functions[schema["name"]] = fn
        self.schemas.append(schema)
        return fn

registry = ToolRegistry()
print("registry ready:", registry.functions)

registry ready: {}


### R2. Tool 1: `search_notes` (lesson 29)
This is Scout's "search". The source project's web search is replaced here because a live search needs
a provider key (RESEARCH.md); Scout searches local notes instead, so it runs with no search key. One
page of matches, long texts shortened, `total` and `next_page`.

In [9]:
def shorten(text: str, limit: int = 120) -> str:
    return text if len(text) <= limit else text[:limit] + f"... [+{len(text) - limit} chars]"

@registry.tool
def search_notes(query: Annotated[str, "a word or short phrase, e.g. 'Mars'; case is ignored"],
                 max_results: Annotated[int, "notes per page, 1 to 10"] = 5,
                 page: Annotated[int, "page number, starting at 1"] = 1) -> dict:
    """Search the user's notes for a word or phrase (case-insensitive). Returns one page of matching
    notes as {id, text} (long texts shortened), the total number of matches and next_page (null when
    there are no more). Use it before answering any question about the notes."""
    q = query.strip().lower()
    if not q or page < 1:
        raise ValueError("query must not be empty and page starts at 1, e.g. query='Mars', page=1")
    hits = [n for n in ALL_NOTES if q in n["text"].lower()]
    size = min(max(max_results, 1), 10)
    start = (page - 1) * size
    return {"total": len(hits), "page": page,
            "results": [{"id": n["id"], "text": shorten(n["text"])} for n in hits[start:start + size]],
            "next_page": page + 1 if start + size < len(hits) else None}

print(search_notes("day on Mars"))

{'total': 1, 'page': 1, 'results': [{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}], 'next_page': None}


### R3. Tool 2: `calculator` (lesson 29)
Parse the text into a syntax tree and walk it with an allow-list of operators. No `eval`, and the
exponent is capped.

In [10]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow}
HOW = "use numbers, + - * / % ** and brackets, e.g. '24*60+39'"

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -evaluate(node.operand)
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        left, right = evaluate(node.left), evaluate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError(f"exponent {right} is too large (max 100)")
        return OPS[type(node.op)](left, right)
    raise ValueError(f"'{ast.unparse(node)[:40]}' is not plain arithmetic: {HOW}")

The tool wraps `evaluate` with a length limit and a readable syntax error.

In [11]:
@registry.tool
def calculator(expression: Annotated[str, "arithmetic only, e.g. '(24*60+39)/60'"]) -> float:
    """Evaluate an arithmetic expression: numbers, + - * / % ** and brackets. Returns the number.
    Use it for every calculation instead of doing arithmetic yourself."""
    if len(expression) > 200:
        raise ValueError("expression too long (max 200 characters); split it into steps")
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError:
        raise ValueError(f"'{expression}' is not valid arithmetic: {HOW}") from None
    return evaluate(tree.body)

print(calculator("24*60+39"), calculator("2**10"))

1479 1024


### R4. Tool 3: `write_file` (lesson 29)
Resolve the path first, then require that its real parent is Scout's folder. An existing file is kept
unless the model sets `overwrite`.

In [12]:
@registry.tool
def write_file(filename: Annotated[str, "a plain file name, e.g. 'mars_day.txt'"],
               text: Annotated[str, "the full text to save"],
               overwrite: Annotated[bool, "true to replace an existing file"] = False) -> str:
    """Save text to a file in Scout's folder. Returns 'saved N characters to NAME'.
    Refuses an existing file unless overwrite is true."""
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve():
        raise ValueError(f"'{filename}' is outside Scout's folder. Use a plain file name, "
                         f"e.g. '{Path(filename).name or 'notes.txt'}'.")
    if target.exists() and not overwrite:
        raise FileExistsError(f"'{filename}' already exists. Pick a new name or set overwrite to true.")
    target.write_text(text)
    return f"saved {len(text)} characters to {target.name}"

print("tools:", [s["name"] for s in registry.schemas])

tools: ['search_notes', 'calculator', 'write_file']


### R5. `run_tool`: never raises (lessons 28, 29)
An unknown name or any exception becomes an error result with `is_error` True. Any result over
`MAX_RESULT_CHARS` is cut, with a note saying how to ask for less.

In [13]:
MAX_RESULT_CHARS = 1500

def run_tool(call: dict, tools: dict) -> dict:
    """Run one tool call. Returns {"content": str, "is_error": bool}; never raises."""
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        result = {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:                                    # any failure becomes an observation
        result = {"content": f"{type(e).__name__}: {e}", "is_error": True}
    extra = len(result["content"]) - MAX_RESULT_CHARS
    if extra > 0:
        result["content"] = (result["content"][:MAX_RESULT_CHARS] + f"... [truncated {extra} "
                             "characters: ask for a smaller page or a narrower query]")
    return result

print(run_tool({"name": "calculator", "args": {"expression": "1479/0"}}, registry.functions))

{'content': 'ZeroDivisionError: division by zero', 'is_error': True}


### R6. `run_agent`: the loop (lesson 26, with lesson 28's error count)
Call the model, append its turn, run every tool call through `run_tool`, append each result with its
call id. No tool calls means the end of turn; running out of steps returns `stop="step_limit"`.

In [14]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 8) -> dict:
    messages, errors = [{"role": "user", "text": question}], 0
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step,
                    "errors": errors, "messages": messages}
        for call in calls:
            result = run_tool(call, tools)
            errors += result["is_error"]
            print(f"step {step}: {call['name']}({call['args']}) -> "
                  f"{'ERROR ' if result['is_error'] else ''}{result['content'][:70]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors, "messages": messages}

### R7. Run the acceptance checks
First the happy path with the trace showing, so you can see the loop work. Then all five checks.

In [15]:
run = run_agent(QUESTION, ScriptedModel(HAPPY), registry.functions, registry.schemas)
print(run["stop"], "| errors:", run["errors"], "|", (SCOUT_DIR / "mars_day.txt").read_text())
check_scout(run_agent, registry)

step 1: search_notes({'query': 'day on Mars'}) -> {"total": 1, "page": 1, "results": [{"id": 3, "text": "A day on Mars l
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}) -> "saved 39 characters to mars_day.txt"
step 4: answer
end_of_turn | errors: 0 | A day on Mars lasts about 1479 minutes.
all checks passed


### R8. Gemini behind the same interface (lessons 26, 28, 29)
Tool declarations come from the schemas. Results go back under `"output"` or `"error"`; a truncated
result is plain text, so `as_output` passes it through as a string. Model turns go back unchanged.

In [16]:
def declarations(schemas: list[dict]) -> list[types.Tool]:
    return [types.Tool(function_declarations=[types.FunctionDeclaration(
        name=s["name"], description=s["description"], parameters_json_schema=s["parameters"])
        for s in schemas])]

def as_output(content: str):
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return content

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                          # keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": as_output(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The model wrapper. It builds the declarations from `tools` on **every** call, so a tool registered
later is sent too. It also keeps each call's input tokens.

In [17]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first. "
          "Use the calculator for arithmetic. If a tool returns an error, fix the call and try again. "
          "Answer in one sentence.")

class GeminiModel:
    def __init__(self):
        self.prompt_tokens = []

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=400,
                                          tools=declarations(tools), automatic_function_calling=NO_AFC)
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        self.prompt_tokens.append(resp.usage_metadata.prompt_token_count)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

### R9. How big is Scout?
The goal said about 150 lines. Count the non-blank, non-comment lines of the functions and class
methods in R1 to R8 (the starter code and the checks are not part of Scout).

`inspect.getsource` cannot read classes defined in a notebook (Python 3.12), so for a class we count its
`class` line plus its methods. Constants such as `OPS` and `SYSTEM` (`JSON_TYPES`, `OPS`, `HOW`,
`MAX_RESULT_CHARS`, `SYSTEM`: 8 lines) are not counted, so the real size is nearer 165 lines. That is
still about the size of the goal.

In [18]:
SCOUT_PARTS = [json_type, schema_from_function, ToolRegistry, shorten, search_notes, evaluate,
               calculator, write_file, run_tool, run_agent, declarations, as_output, to_contents, GeminiModel]

def code_lines(obj) -> int:
    if inspect.isclass(obj):          # getsource fails on notebook classes: class line + methods instead
        return 1 + sum(code_lines(f) for f in vars(obj).values() if inspect.isfunction(f))
    return sum(1 for line in inspect.getsource(obj).splitlines()
               if line.strip() and not line.strip().startswith("#"))

for part in SCOUT_PARTS:
    print(f"{part.__name__:21} {code_lines(part):3}")
print(f"{'Scout, total':21} {sum(code_lines(p) for p in SCOUT_PARTS):3} lines")

json_type               6
schema_from_function   16
ToolRegistry           11
shorten                 2
search_notes           16
evaluate               11
calculator             11
write_file             14
run_tool               15
run_agent              17
declarations            4
as_output               5
to_contents            16
GeminiModel            13
Scout, total          157 lines


### R10. Scout on the real model
Same loop, same tools, Gemini choosing every call. The folder is emptied first, so the save is new.

In [19]:
clear_folder()
gemini = GeminiModel()
run = run_agent(QUESTION, gemini, registry.functions, registry.schemas)
print(run["stop"], "| errors:", run["errors"], "|", run["answer"])
print("input tokens per call:", gemini.prompt_tokens)
print("saved:", sorted(p.name for p in SCOUT_DIR.iterdir()))

step 1: search_notes({'query': 'Mars'}) -> {"total": 34, "page": 1, "results": [{"id": 1, "text": "Mars has two s
step 2: calculator({'expression': '24*60+39'}) -> 1479


step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars is 1479 minutes long.', 'overwrite': True}) -> "saved 35 characters to mars_day.txt"
step 4: answer
end_of_turn | errors: 0 | A day on Mars is 1479 minutes long, and this has been saved to mars_day.txt.
input tokens per call: [340, 476, 500, 548]


saved: ['mars_day.txt']


Gemini searched for "Mars" (34 matches; the day note is on page 1), calculated `24*60+39`, and saved
the answer, setting `overwrite` to true on its own. Four model calls, no errors. Input grew from 340 to
548 tokens, because each call resends the whole history (lesson 08).

## Exit test
### Part 1: an extra tool without touching the loop
We take a fingerprint (a SHA-256 hash) of the loop's source code and of `run_tool` now, add the tool,
and compare afterwards. A hash is a short code. Changing even one character of the source gives a
different code. This checks the text of `run_agent` and `run_tool` only. The extra tool is
`read_file`: Scout reads back a file it saved earlier.

In [20]:
def fingerprint(*parts) -> str:
    return hashlib.sha256("".join(inspect.getsource(p) for p in parts).encode()).hexdigest()[:12]

loop_before = fingerprint(run_agent, run_tool)
print("loop fingerprint before:", loop_before, "| tools:", list(registry.functions))

source = inspect.getsource(run_agent)                          # demo: one character changed
changed = source.replace("max_steps: int = 8", "max_steps: int = 9")
print("one character changed:  ", hashlib.sha256(changed.encode()).hexdigest()[:12],
      "vs", hashlib.sha256(source.encode()).hexdigest()[:12])

loop fingerprint before: 80ef4d215a22 | tools: ['search_notes', 'calculator', 'write_file']
one character changed:   9418e9c7ed9f vs 2f2ee7d4dbbf


The tool is one decorated function. A missing file raises `FileNotFoundError` with an actionable
message: it lists the files that do exist and says to retry with one of them.

In [21]:
@registry.tool
def read_file(filename: Annotated[str, "a plain file name, e.g. 'mars_day.txt'"]) -> str:
    """Read back a text file that Scout saved earlier in its folder. Returns the full text."""
    target = (SCOUT_DIR / filename).resolve()
    if target.parent != SCOUT_DIR.resolve() or not target.is_file():
        saved = sorted(p.name for p in SCOUT_DIR.iterdir() if p.is_file())
        raise FileNotFoundError(f"no file '{filename}' in Scout's folder. Saved files: "
                                f"{', '.join(saved) or 'none'}. Retry with one of those names.")
    return target.read_text()

loop_after = fingerprint(run_agent, run_tool)
print("loop fingerprint after: ", loop_after, "| unchanged:", loop_before == loop_after)
print("tools now:", [s["name"] for s in registry.schemas])

loop fingerprint after:  80ef4d215a22 | unchanged: True
tools now: ['search_notes', 'calculator', 'write_file', 'read_file']


The loop is the same code and the registry has four tools. A matching hash only shows the text did not
change. The real proof is that the new tool works end to end: the acceptance checks below still pass
with it in place, and Part 2 runs `read_file` through the unchanged loop.

In [22]:
check_scout(run_agent, registry)

all checks passed


### Part 2: recover from a tool that raises
First with the fake. Its first call asks for `mars-day.txt` (a dash, not an underscore). The tool
raises; `run_tool` turns that into an error result; the fake reads the list of saved files from the
message and retries with the real name, as a model would.

In [23]:
write_file("mars_day.txt", "A day on Mars lasts about 1479 minutes.", overwrite=True)

def retry_from_list(messages):
    listed = re.search(r"Saved files: (.+?)\. Retry", messages[-1]["content"]).group(1)
    return {"tool_calls": [{"name": "read_file", "args": {"filename": listed.split(", ")[0]}}]}

READ_SCRIPT = [{"tool_calls": [{"name": "read_file", "args": {"filename": "mars-day.txt"}}]},
               retry_from_list, lambda m: {"text": f"The file says: {last_result(m)}"}]
run = run_agent("What does my file mars-day.txt say?", ScriptedModel(READ_SCRIPT),
                registry.functions, registry.schemas)
print(run["stop"], "| errors:", run["errors"], "|", run["answer"])

step 1: read_file({'filename': 'mars-day.txt'}) -> ERROR FileNotFoundError: no file 'mars-day.txt' in Scout's folder. Saved fil
step 2: read_file({'filename': 'mars_day.txt'}) -> "A day on Mars lasts about 1479 minutes."
step 3: answer
end_of_turn | errors: 1 | The file says: A day on Mars lasts about 1479 minutes.


Now the real model, same question. Gemini sees four tools, because `GeminiModel` reads the registry's
schemas on every call. Nobody told it about the dash.

In [24]:
if not (SCOUT_DIR / "mars_day.txt").exists():                 # setup, in case the cell above was skipped
    write_file("mars_day.txt", "A day on Mars lasts about 1479 minutes.")
run = run_agent("What does my file mars-day.txt say?", GeminiModel(), registry.functions, registry.schemas)
print(run["stop"], "| errors:", run["errors"], "|", run["answer"])
for m in run["messages"]:
    if m["role"] == "tool" and m["is_error"]:
        print("error the model read:", m["content"])

step 1: read_file({'filename': 'mars-day.txt'}) -> ERROR FileNotFoundError: no file 'mars-day.txt' in Scout's folder. Saved fil


step 2: read_file({'filename': 'mars_day.txt'}) -> "A day on Mars lasts about 1479 minutes."
step 3: answer
end_of_turn | errors: 1 | Your file mars_day.txt states that a day on Mars lasts about 1479 minutes.
error the model read: FileNotFoundError: no file 'mars-day.txt' in Scout's folder. Saved files: mars_day.txt. Retry with one of those names.


Gemini first asked for `mars-day.txt`, exactly as typed. The error listed `mars_day.txt`; the next call
used that name, and the answer came from the file. One error, then `end_of_turn`. The loop did nothing
special: the recovery came from the message alone. That is the second half of the exit test.

## At work
- **Loops are unit-tested with a scripted fake**, as `check_scout` does: no key, no cost, never
  flaky. RESEARCH.md §7 pairs these with tool tests and cassette tests. → lesson 46
- **A tool change is proven with an eval, not one run.** Anthropic's "Writing tools for agents" reports
  that one description fix changed Claude's searches; teams measure such changes over many
  questions. → lesson 48
- **Notes and files are untrusted.** What `read_file` returns enters the model's context, so a note
  saying "ignore your instructions" is an attack. → lesson 51
- **`max_steps` is not a budget.** Production runs also cap tokens and cost, and log each step's
  tool, arguments and result size. → lessons 49 and 54

## What just happened
You built **Scout v1**: lesson 18's Scout chat, now a tool-using agent (lesson 55 builds Scout v2,
lesson 60 Scout v3).
- The registry built four schemas from typed, documented functions; the loop read them without knowing
  their names.
- `run_tool` turned an unknown tool, a bad expression, a path outside the folder and a missing file
  into error results; nothing crashed.
- `check_scout` passed five fake scenarios, before and after the extra tool, `read_file`, was added.
- Scout came to 157 counted lines (functions and class methods, constants excluded; R9), Gemini wrapper
  included.
- The loop's fingerprint did not change when `read_file` was added: the exit test's first half.
- The fake and Gemini both read `read_file`'s error, retried with the listed name and answered: the
  second half.

## Common mistakes
**1. A tool that returns something JSON cannot encode.** This `write_file` returns the `Path` it wrote.
The file is saved, but `json.dumps` fails inside `run_tool`, so the model is told the tool **failed**.
The model sees `TypeError: Object of type PosixPath is not JSON serializable`. It may save again, or
tell the user the save did not work.

In [25]:
def write_file_path(filename: str, text: str) -> Path:
    target = SCOUT_DIR / Path(filename).name
    target.write_text(text)
    return target                                             # a Path, not a string

result = run_tool({"name": "write_file", "args": {"filename": "moons.txt", "text": "2 moons"}},
                  {"write_file": write_file_path})
print("saved on disk:", (SCOUT_DIR / "moons.txt").exists())
assert not result["is_error"], f"the model was told: {result['content']}"

saved on disk: True


AssertionError: the model was told: TypeError: Object of type PosixPath is not JSON serializable

Fix: return plain JSON types (`str`, numbers, lists, dicts), as the real `write_file` does with its
`"saved N characters to NAME"` string. A test that calls each tool through `run_tool` catches this.

In [26]:
print(run_tool({"name": "write_file", "args": {"filename": "moons2.txt", "text": "2 moons"}},
               registry.functions))

{'content': '"saved 7 characters to moons2.txt"', 'is_error': False}


**2. Building the tool list once, in `__init__`.** It looks tidy, but the model then sees the tools
that existed when it was created. A tool added later is in the registry and invisible to the model.

In [27]:
class GeminiModelFrozen(GeminiModel):
    def __init__(self, schemas):
        super().__init__()
        self.decls = declarations(list(schemas))              # copied once, at creation

frozen = GeminiModelFrozen(registry.schemas[:3])              # created before read_file existed
print("model was built with:", [d.name for d in frozen.decls[0].function_declarations])

model was built with: ['search_notes', 'calculator', 'write_file']


The registry has four tools, the frozen model knows three.

In [28]:
sent = [d.name for d in frozen.decls[0].function_declarations]
assert "read_file" in sent, f"registry has {list(registry.functions)}, but the model gets {sent}"

AssertionError: registry has ['search_notes', 'calculator', 'write_file', 'read_file'], but the model gets ['search_notes', 'calculator', 'write_file']

Fix: build the declarations from the `tools` argument on every call, as `GeminiModel.__call__` does.
The loop passes the registry's current schemas each step, so new tools appear at once.

In [29]:
print("sent on every call:", [d.name for d in declarations(registry.schemas)[0].function_declarations])

sent on every call: ['search_notes', 'calculator', 'write_file', 'read_file']


## Try it
Add a fifth tool, `convert_time(value: float, from_unit: str, to_unit: str)`, for `seconds`, `minutes`,
`hours` and `days`. For any other unit, raise a `ValueError` that lists the allowed units. Write a fake
script that first sends `from_unit="hrs"`, then retries with `"hours"` after the error, and answers
from the result (24.65 hours is 1479 minutes). Check `errors == 1`, the answer contains 1479, and the
loop's fingerprint is still `loop_before`.

In [30]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [31]:
llm_replay.stop()

## Key terms
- **open to new tools** — a loop that gets a new tool by registering a function, with no change to the loop's code; the first half of this checkpoint's exit test.
- **loop fingerprint** — a hash of the loop's source code, taken before and after a change, to prove the change did not touch the loop.
- **acceptance check** — a test the finished program must pass, here `check_scout`: five scripted runs covering the happy path, a raising tool, an unknown tool, a path outside the folder and the step limit.